# Optima Life Growth Strategy Analysis

Analyzes how a fictional multi-product SaaS business should balance new customer acquisition with retention and expansion of its existing subscriber base.

**Portfolio note:** This analysis was originally executed in a university-hosted Snowflake environment using a fictional SaaS dataset. The raw course dataset and hosted environment are not redistributed here. This notebook preserves the analytical workflow, SQL/Python code, methodology, and conclusions; the repository README summarizes the verified executed results.


In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE VIEW ARR_ROW_LEVEL AS
SELECT
    *,
    CASE
        WHEN DATEDIFF(
            DAY,
            SUBSCRIPTION_START_DATE,
            SUBSCRIPTION_END_DATE
        ) <> 0
        THEN SUBSCRIPTION_AMOUNT * 365
             / DATEDIFF(
                 DAY,
                 SUBSCRIPTION_START_DATE,
                 SUBSCRIPTION_END_DATE
             )
        ELSE 0
    END AS ROW_ARR
FROM SUBSCRIPTION_DATA.PROJECT_DATA.OL_SUBSCRIPTIONS;


In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE VIEW ARR_ROW_YEAR AS
SELECT
    A.CUSTOMER_ID,
    A.PRODUCT,
    X.YEAR_DATES,

    SUM(
        CASE
            WHEN X.YEAR_DATES BETWEEN
                 A.SUBSCRIPTION_START_DATE
                 AND A.SUBSCRIPTION_END_DATE
            THEN A.ROW_ARR
            ELSE 0
        END
    ) AS YEAR_BEGIN_ARR

FROM ARR_ROW_LEVEL A

CROSS JOIN
(
    SELECT COLUMN1::DATE AS YEAR_DATES
    FROM VALUES
        ('2019-12-01'),
        ('2020-12-01'),
        ('2021-12-01'),
        ('2022-12-01'),
        ('2023-12-01'),
        ('2024-12-01')
) X

GROUP BY
    A.CUSTOMER_ID,
    A.PRODUCT,
    X.YEAR_DATES;


In [ ]:
%%sql -r dataframe_5

SELECT
    PRODUCT,
    YEAR_DATES,

    COUNT(
        DISTINCT CASE
            WHEN YEAR_BEGIN_ARR > 0 THEN CUSTOMER_ID
        END
    ) AS ACTIVE_CUSTOMER_COUNT,

    SUM(YEAR_BEGIN_ARR) AS BEGINNING_ARR

FROM ARR_ROW_YEAR

GROUP BY
    PRODUCT,
    YEAR_DATES

ORDER BY
    PRODUCT,
    YEAR_DATES;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE VIEW ARR_SUMMARY_PRODUCT AS

WITH customer_rollup AS
(
    SELECT
        PRODUCT,
        CUSTOMER_ID,
        YEAR_DATES,
        SUM(YEAR_BEGIN_ARR) AS YR_BEG_ARR
    FROM ARR_ROW_YEAR
    GROUP BY
        PRODUCT,
        CUSTOMER_ID,
        YEAR_DATES
),

customer_rollup_w_prev AS
(
    SELECT
        *,
        LEAD(YR_BEG_ARR) OVER
        (
            PARTITION BY PRODUCT, CUSTOMER_ID
            ORDER BY YEAR_DATES
        ) AS NXT_PERIOD_ARR
    FROM customer_rollup
),

arr_variations AS
(
    SELECT
        *,

        CASE
            WHEN YR_BEG_ARR = 0
                 AND NXT_PERIOD_ARR > 0
            THEN NXT_PERIOD_ARR
            ELSE 0
        END AS NEW_ARR,

        CASE
            WHEN YR_BEG_ARR > 0
                 AND NXT_PERIOD_ARR > YR_BEG_ARR
            THEN NXT_PERIOD_ARR - YR_BEG_ARR
            ELSE 0
        END AS EXPANSION_ARR,

        CASE
            WHEN YR_BEG_ARR > 0
                 AND NXT_PERIOD_ARR > 0
                 AND NXT_PERIOD_ARR < YR_BEG_ARR
            THEN NXT_PERIOD_ARR - YR_BEG_ARR
            ELSE 0
        END AS CONTRACTION_ARR,

        CASE
            WHEN YR_BEG_ARR > 0
                 AND NXT_PERIOD_ARR = 0
            THEN NXT_PERIOD_ARR - YR_BEG_ARR
            ELSE 0
        END AS CHURN_ARR

    FROM customer_rollup_w_prev
)

SELECT
    PRODUCT,
    YEAR_DATES,

    SUM(YR_BEG_ARR) AS BEGINNING_ARR,
    SUM(NEW_ARR) AS NEW_ARR,
    SUM(EXPANSION_ARR) AS EXPANSION_ARR,
    SUM(CONTRACTION_ARR) AS CONTRACTION_ARR,
    SUM(CHURN_ARR) AS CHURN_ARR,

    CASE
        WHEN YEAR_DATES = DATE '2024-12-01'
        THEN NULL
        ELSE SUM(
            YR_BEG_ARR
            + NEW_ARR
            + EXPANSION_ARR
            + CONTRACTION_ARR
            + CHURN_ARR
        )
    END AS ENDING_ARR

FROM arr_variations

GROUP BY
    PRODUCT,
    YEAR_DATES

ORDER BY
    PRODUCT,
    YEAR_DATES;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE VIEW PRODUCT_GROWTH_METRICS AS

WITH growth_metrics AS
(
    SELECT
        *,
        LAG(BEGINNING_ARR) OVER
        (
            PARTITION BY PRODUCT
            ORDER BY YEAR_DATES
        ) AS PREV_BEGINNING_ARR

    FROM ARR_SUMMARY_PRODUCT
)

SELECT

    PRODUCT,
    YEAR_DATES,

    BEGINNING_ARR,
    NEW_ARR,
    EXPANSION_ARR,
    CONTRACTION_ARR,
    CHURN_ARR,
    ENDING_ARR,

    /* Year-over-year ARR change */
    BEGINNING_ARR - PREV_BEGINNING_ARR
        AS Y_O_Y_CHANGE,

    /* Year-over-year ARR percentage change */
    (
        BEGINNING_ARR - PREV_BEGINNING_ARR
    ) * 100
    / NULLIF(PREV_BEGINNING_ARR,0)
        AS Y_O_Y_CHANGE_PCT,

    /* ARR change metrics relative to the beginning ARR */
    NEW_ARR * 100
        / NULLIF(BEGINNING_ARR,0)
        AS NEW_PCT,

    EXPANSION_ARR * 100
        / NULLIF(BEGINNING_ARR,0)
        AS EXPANSION_PCT,

    CONTRACTION_ARR * 100
        / NULLIF(BEGINNING_ARR,0)
        AS CONTRACTION_PCT,

    CHURN_ARR * 100
        / NULLIF(BEGINNING_ARR,0)
        AS CHURN_PCT,

    /* Share of positive ARR growth from new customers */
    CASE
        WHEN NEW_ARR + EXPANSION_ARR = 0
        THEN 0
        ELSE NEW_ARR * 100
             / (NEW_ARR + EXPANSION_ARR)
    END AS LANDED_PCT_OF_NEW,

    /* Share of positive ARR growth from existing customers */
    CASE
        WHEN NEW_ARR + EXPANSION_ARR = 0
        THEN 0
        ELSE EXPANSION_ARR * 100
             / (NEW_ARR + EXPANSION_ARR)
    END AS EXPANDED_PCT_OF_NEW

FROM growth_metrics

ORDER BY
    PRODUCT,
    YEAR_DATES;


--6. Preview
SELECT *
FROM PRODUCT_GROWTH_METRICS
ORDER BY
    PRODUCT,
    YEAR_DATES;

In [ ]:
growth_df = session.sql("""
    SELECT *
    FROM PRODUCT_GROWTH_METRICS
    ORDER BY PRODUCT, YEAR_DATES
""").to_pandas()

growth_df.head()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import PercentFormatter

growth_df["YEAR_DATES"] = pd.to_datetime(growth_df["YEAR_DATES"])
growth_df["YEAR"] = growth_df["YEAR_DATES"].dt.year

growth_df.info()

## Building on Previous Findings

The ARR summary table above provides a consolidated view of the product-level performance previously analyzed in Assignment 1.

In [ ]:
#Chart 1: New ARR vs. Expansion ARR by Product( Cumilative across entrire period)

df_growth = session.sql("""
    SELECT
        PRODUCT,
        SUM(NEW_ARR) AS NEW_ARR,
        SUM(EXPANSION_ARR) AS EXPANSION_ARR
    FROM PRODUCT_GROWTH_METRICS
    WHERE YEAR_DATES < '2024-12-01'
    GROUP BY PRODUCT
    ORDER BY PRODUCT
""").to_pandas()

ax = df_growth.plot(
    x="PRODUCT",
    y=["NEW_ARR", "EXPANSION_ARR"],
    kind="bar",
    figsize=(11, 6),
    width=0.75
)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"${value / 1_000_000:.1f}M")
)

ax.set_title("New Customer ARR vs. Existing Customer Expansion by Product")
ax.set_xlabel("Product")
ax.set_ylabel("Cumulative ARR Contribution")
ax.legend(["New ARR", "Expansion ARR"], title="Growth Source")
ax.grid(axis="y", alpha=0.3)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

The above chart shows the cumulative contribution of New ARR and Expansion ARR across the entire analysis period. It helps us understand how each product has historically generated growth. It is clear that New ARR has been the primary driver of growth across all products, while Expansion ARR has made a smaller but still positive contribution.


In [ ]:
%%sql -r dataframe_7
--Evolution of New ARR and Expansion ARR Over Time

SELECT
    YEAR_DATES,
    SUM(NEW_ARR) AS NEW_ARR,
    SUM(EXPANSION_ARR) AS EXPANSION_ARR
FROM PRODUCT_GROWTH_METRICS
WHERE YEAR_DATES < '2024-12-01'
GROUP BY YEAR_DATES
ORDER BY YEAR_DATES
;

New ARR increased through 2021 before declining in 2022 and 2023. Expansion ARR followed the opposite pattern, increasing consistently throughout the analysis period.

Customer acquisition remains the larger source of positive ARR, but its contribution has weakened since its 2021 peak. Meanwhile, existing customers are generating a growing amount of expansion revenue. The next analysis examines how the share of positive growth coming from each source has changed over time.


In [ ]:
%%sql -r dataframe_8
SELECT
    YEAR_DATES,
    SUM(NEW_ARR) AS NEW_ARR,
    SUM(EXPANSION_ARR) AS EXPANSION_ARR
FROM PRODUCT_GROWTH_METRICS
WHERE YEAR_DATES < '2024-12-01'
GROUP BY YEAR_DATES
ORDER BY YEAR_DATES;

In [ ]:
# Share of Growth from New ARR and Expansion ARR

df = session.sql("""
SELECT
    YEAR_DATES,
    SUM(NEW_ARR) AS NEW_ARR,
    SUM(EXPANSION_ARR) AS EXPANSION_ARR
FROM PRODUCT_GROWTH_METRICS
GROUP BY YEAR_DATES
ORDER BY YEAR_DATES
""").to_pandas()

# Convert YEAR_DATES to pandas datetime
df["YEAR_DATES"] = pd.to_datetime(df["YEAR_DATES"])

# Remove the incomplete December 2024 period
df = df[df["YEAR_DATES"] < pd.Timestamp("2024-12-01")]

df["YEAR"] = df["YEAR_DATES"].dt.year

df["TOTAL_POSITIVE_ARR"] = df["NEW_ARR"] + df["EXPANSION_ARR"]
df["NEW_PCT"] = df["NEW_ARR"] / df["TOTAL_POSITIVE_ARR"]
df["EXPANSION_PCT"] = df["EXPANSION_ARR"] / df["TOTAL_POSITIVE_ARR"]

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    df["YEAR"],
    df["NEW_PCT"],
    label="New ARR"
)

ax.bar(
    df["YEAR"],
    df["EXPANSION_PCT"],
    bottom=df["NEW_PCT"],
    label="Expansion ARR"
)

ax.yaxis.set_major_formatter(PercentFormatter(1))

ax.set_title("Share of Growth from New ARR and Expansion ARR")
ax.set_xlabel("Year")
ax.set_ylabel("Share of Total Positive ARR")
ax.set_xticks(df["YEAR"])
ax.legend(title="Growth Source")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

 The following step is to determine whether customers who expand their subscriptions actually generate greater long-term value than those who never expand. 

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE VIEW CUSTOMER_YEAR_ARR AS

SELECT
    CUSTOMER_ID,
    YEAR_DATES,
    SUM(YEAR_BEGIN_ARR) AS TOTAL_BEGINNING_ARR
FROM ARR_ROW_YEAR
GROUP BY
    CUSTOMER_ID,
    YEAR_DATES;

SELECT *
FROM CUSTOMER_YEAR_ARR
ORDER BY CUSTOMER_ID, YEAR_DATES
LIMIT 20;

In [ ]:
%%sql -r dataframe_11
-- Identify true year-over-year customer expansion

CREATE OR REPLACE VIEW CUSTOMER_EXPANSION_HISTORY AS

WITH customer_arr_with_previous AS (
    SELECT
        CUSTOMER_ID,
        YEAR_DATES,
        TOTAL_BEGINNING_ARR,

        LAG(TOTAL_BEGINNING_ARR) OVER (
            PARTITION BY CUSTOMER_ID
            ORDER BY YEAR_DATES
        ) AS PREVIOUS_YEAR_ARR

    FROM CUSTOMER_YEAR_ARR
)

SELECT
    CUSTOMER_ID,
    YEAR_DATES,
    TOTAL_BEGINNING_ARR,
    PREVIOUS_YEAR_ARR,

    CASE
        WHEN PREVIOUS_YEAR_ARR > 0
             AND TOTAL_BEGINNING_ARR > PREVIOUS_YEAR_ARR
        THEN 1
        ELSE 0
    END AS EXPANDED_THIS_YEAR,

    CASE
        WHEN PREVIOUS_YEAR_ARR > 0
             AND TOTAL_BEGINNING_ARR > PREVIOUS_YEAR_ARR
        THEN TOTAL_BEGINNING_ARR - PREVIOUS_YEAR_ARR
        ELSE 0
    END AS EXPANSION_AMOUNT

FROM customer_arr_with_previous;

SELECT *
FROM CUSTOMER_EXPANSION_HISTORY
ORDER BY CUSTOMER_ID, YEAR_DATES
LIMIT 50;

In [ ]:
%%sql -r dataframe_9
-- Classify each customer as Expanded or Never Expanded

CREATE OR REPLACE VIEW CUSTOMER_EXPANSION_FLAG AS

SELECT
    CUSTOMER_ID,

    MAX(EXPANDED_THIS_YEAR) AS EVER_EXPANDED,

    CASE
        WHEN MAX(EXPANDED_THIS_YEAR) = 1
        THEN 'Expanded'
        ELSE 'Never Expanded'
    END AS EXPANSION_GROUP,

    SUM(EXPANSION_AMOUNT) AS TOTAL_EXPANSION_AMOUNT,

    COUNT_IF(EXPANDED_THIS_YEAR = 1) AS NUMBER_OF_EXPANSION_YEARS

FROM CUSTOMER_EXPANSION_HISTORY

GROUP BY CUSTOMER_ID;

SELECT *
FROM CUSTOMER_EXPANSION_FLAG
ORDER BY CUSTOMER_ID
LIMIT 50;

In [ ]:
%%sql -r dataframe_12
-- Query 4: Calculate value and retention measures for each customer

CREATE OR REPLACE VIEW CUSTOMER_VALUE_SUMMARY AS

WITH customer_metrics AS (
    SELECT
        CUSTOMER_ID,

        MIN(
            CASE
                WHEN TOTAL_BEGINNING_ARR > 0
                THEN YEAR_DATES
            END
        ) AS FIRST_ACTIVE_YEAR,

        MAX(
            CASE
                WHEN TOTAL_BEGINNING_ARR > 0
                THEN YEAR_DATES
            END
        ) AS LAST_ACTIVE_YEAR,

        COUNT_IF(TOTAL_BEGINNING_ARR > 0) AS ACTIVE_YEARS,

        SUM(TOTAL_BEGINNING_ARR) AS TOTAL_ARR_GENERATED,

        AVG(
            CASE
                WHEN TOTAL_BEGINNING_ARR > 0
                THEN TOTAL_BEGINNING_ARR
            END
        ) AS AVERAGE_ACTIVE_YEAR_ARR,

        MAX_BY(
            TOTAL_BEGINNING_ARR,
            YEAR_DATES
        ) AS FINAL_YEAR_ARR

    FROM CUSTOMER_YEAR_ARR

    GROUP BY CUSTOMER_ID
)

SELECT
    m.CUSTOMER_ID,
    f.EVER_EXPANDED,
    f.EXPANSION_GROUP,
    f.TOTAL_EXPANSION_AMOUNT,
    f.NUMBER_OF_EXPANSION_YEARS,

    m.FIRST_ACTIVE_YEAR,
    m.LAST_ACTIVE_YEAR,
    m.ACTIVE_YEARS,

    DATEDIFF(
        YEAR,
        m.FIRST_ACTIVE_YEAR,
        m.LAST_ACTIVE_YEAR
    ) + 1 AS CUSTOMER_LIFETIME_YEARS,

    m.TOTAL_ARR_GENERATED,
    m.AVERAGE_ACTIVE_YEAR_ARR,
    m.FINAL_YEAR_ARR,

    CASE
        WHEN m.FINAL_YEAR_ARR > 0
        THEN 0
        ELSE 1
    END AS CHURNED

FROM customer_metrics m

INNER JOIN CUSTOMER_EXPANSION_FLAG f
    ON m.CUSTOMER_ID = f.CUSTOMER_ID;


    SELECT *
FROM CUSTOMER_VALUE_SUMMARY
ORDER BY CUSTOMER_ID
LIMIT 50;


In [ ]:
# Read data from the SQL query
query = """
SELECT
    EXPANSION_GROUP,
    ROUND(AVG(CUSTOMER_LIFETIME_YEARS),2) AS AVG_LIFETIME,
    ROUND(AVG(AVERAGE_ACTIVE_YEAR_ARR),2) AS AVG_ANNUAL_ARR,
    ROUND(AVG(FINAL_YEAR_ARR),2) AS AVG_FINAL_ARR,
    ROUND(100 * AVG(CHURNED),2) AS CHURN_RATE
FROM CUSTOMER_VALUE_SUMMARY
GROUP BY EXPANSION_GROUP
ORDER BY EXPANSION_GROUP;
"""

df = session.sql(query).to_pandas()

# Plot
ax = df.set_index("EXPANSION_GROUP").plot(
    kind="bar",
    figsize=(9,6),
    width=0.75
)

plt.title("Customer Value Comparison: Expanded vs. Never Expanded", fontsize=14)
plt.xlabel("")
plt.ylabel("Average Value")
plt.xticks(rotation=0)
plt.legend(title="")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The customer-level analysis reveals a clear relationship between expanding existing customer relationships and long-term customer value. Compared to customers who never expanded their subscriptions, expanded customers remained active for longer, generated higher recurring revenue, and experienced significantly lower churn. Together, these findings suggest that extracting more value from the existing subscriber base through both expansion and stronger customer retention is associated with greater long-term customer value.

The following analysis compares only customers who had at least two active years, ensuring both groups had a similar opportunity to expand. This validation provides a fairer assessment of whether the relationship between existing customer value and long-term performance still holds.

In [ ]:
%%sql -r dataframe_15
--Compare customers who had the same opportunity to expand

CREATE OR REPLACE VIEW MATCHED_CUSTOMER_COMPARISON AS

WITH eligible_customers AS (

    SELECT
        CUSTOMER_ID
    FROM CUSTOMER_VALUE_SUMMARY
    WHERE ACTIVE_YEARS >= 2

)

SELECT
    c.CUSTOMER_ID,
    c.EXPANSION_GROUP,
    c.CUSTOMER_LIFETIME_YEARS,
    c.TOTAL_ARR_GENERATED,
    c.AVERAGE_ACTIVE_YEAR_ARR,
    c.FINAL_YEAR_ARR,
    c.CHURNED

FROM CUSTOMER_VALUE_SUMMARY c

INNER JOIN eligible_customers e
    ON c.CUSTOMER_ID = e.CUSTOMER_ID;


In [ ]:
%%sql -r dataframe_16
-- Query 8: Compare matched customers

SELECT

    EXPANSION_GROUP,

    COUNT(*) AS CUSTOMERS,

    ROUND(AVG(CUSTOMER_LIFETIME_YEARS),2) AS AVG_LIFETIME,

    ROUND(AVG(TOTAL_ARR_GENERATED),2) AS AVG_TOTAL_ARR,

    ROUND(AVG(AVERAGE_ACTIVE_YEAR_ARR),2) AS AVG_ANNUAL_ARR,

    ROUND(AVG(FINAL_YEAR_ARR),2) AS AVG_FINAL_ARR,

    ROUND(100.0*AVG(CHURNED),2) AS CHURN_RATE

FROM MATCHED_CUSTOMER_COMPARISON

GROUP BY EXPANSION_GROUP;

The validation analysis shows that the relationship remains even when the comparison is limited to customers with at least two active years. Expanded customers averaged 4.15 active years compared with 3.17 years for customers who never expanded. They also generated higher recurring revenue and had a substantially lower churn rate of 14.2%, compared with 41.2% for customers who never expanded.




In [ ]:
%%sql -r dataframe_13
--Producat growth strategy matrix 
CREATE OR REPLACE VIEW PRODUCT_STRATEGY_SUMMARY AS

WITH product_year AS (
    SELECT
        PRODUCT,
        YEAR_DATES,
        SUM(NEW_ARR) AS NEW_ARR,
        SUM(EXPANSION_ARR) AS EXPANSION_ARR,
        SUM(ABS(CONTRACTION_ARR)) AS CONTRACTION_LOSS,
        SUM(ABS(CHURN_ARR)) AS CHURN_LOSS
    FROM PRODUCT_GROWTH_METRICS
    WHERE YEAR_DATES < '2024-12-01'
    GROUP BY PRODUCT, YEAR_DATES
),

product_summary AS (
    SELECT
        PRODUCT,

        AVG(NEW_ARR) AS AVG_NEW_ARR,
        AVG(EXPANSION_ARR) AS AVG_EXPANSION_ARR,
        AVG(CONTRACTION_LOSS + CHURN_LOSS) AS AVG_RETENTION_LOSS,

        REGR_SLOPE(
            NEW_ARR,
            YEAR(YEAR_DATES)
        ) AS NEW_ARR_TREND,

        REGR_SLOPE(
            EXPANSION_ARR,
            YEAR(YEAR_DATES)
        ) AS EXPANSION_ARR_TREND

    FROM product_year
    GROUP BY PRODUCT
)

SELECT
    PRODUCT,
    ROUND(AVG_NEW_ARR, 2) AS AVG_NEW_ARR,
    ROUND(AVG_EXPANSION_ARR, 2) AS AVG_EXPANSION_ARR,
    ROUND(AVG_RETENTION_LOSS, 2) AS AVG_RETENTION_LOSS,
    ROUND(NEW_ARR_TREND, 2) AS NEW_ARR_TREND,
    ROUND(EXPANSION_ARR_TREND, 2) AS EXPANSION_ARR_TREND,

    CASE
        WHEN NEW_ARR_TREND > 0
             AND AVG_NEW_ARR > AVG_EXPANSION_ARR
            THEN 'Continue New ARR Focus'

        WHEN EXPANSION_ARR_TREND > 0
             OR AVG_RETENTION_LOSS > AVG_EXPANSION_ARR
            THEN 'Retention and Expansion Focus'

        ELSE 'Balanced Strategy'
    END AS RECOMMENDED_FOCUS

FROM product_summary
ORDER BY PRODUCT;

In [ ]:
import matplotlib.pyplot as plt

query = """
SELECT
    PRODUCT,
    AVG_NEW_ARR,
    AVG_EXPANSION_ARR,
    AVG_RETENTION_LOSS,
    NEW_ARR_TREND,
    EXPANSION_ARR_TREND,
    RECOMMENDED_FOCUS
FROM PRODUCT_STRATEGY_SUMMARY
ORDER BY PRODUCT;
"""

strategy_df = session.sql(query).to_pandas()

plt.figure(figsize=(10, 7))

plt.scatter(
    strategy_df["AVG_NEW_ARR"],
    strategy_df["AVG_EXPANSION_ARR"],
    s=180
)

for _, row in strategy_df.iterrows():
    plt.annotate(
        row["PRODUCT"],
        (row["AVG_NEW_ARR"], row["AVG_EXPANSION_ARR"]),
        xytext=(6, 6),
        textcoords="offset points"
    )

x_mid = strategy_df["AVG_NEW_ARR"].median()
y_mid = strategy_df["AVG_EXPANSION_ARR"].median()

plt.axvline(x=x_mid, linestyle="--", alpha=0.5)
plt.axhline(y=y_mid, linestyle="--", alpha=0.5)

plt.title("Product Growth Strategy Matrix")
plt.xlabel("Average New ARR")
plt.ylabel("Average Expansion ARR")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Final Growth Strategy Recommendation

The analysis shows that OptimaLife should maintain customer acquisition while shifting more emphasis toward retention and expansion. New ARR remains the largest source of positive growth, but it peaked in 2021 and has declined since, while Expansion ARR increased its share from approximately 1.5% in 2019 to 16.3% in 2023. Customers who expanded were also associated with longer relationships, higher recurring revenue, and substantially lower churn.

OptimaLife should therefore pursue three coordinated actions:

1. **Maintain targeted acquisition rather than broad acquisition across every product.** Premium Health should remain the primary new-ARR opportunity, with selective acquisition for Daily Fitness.

2. **Increase investment in customer retention and expansion.** Healthy Meals and Daily Fitness provide strong foundations for cross-selling, upgrades, and deeper customer relationships.

3. **Turn around underperforming products before increasing acquisition spending.** Mindful Living and Wellness Tracker should undergo repositioning, bundling, or feature-consolidation tests before additional growth investment or discontinuation decisions.

The overall strategy should be to protect the existing customer base, expand the value of retained relationships, and acquire new customers selectively where current product performance supports continued growth.
